# Butters — Live ML Demo (Colab)

Move your laptop around. The browser webcam streams into Colab, YOLO detects trash, and the ApproachController emits an action (FORWARD / LEFT / RIGHT / STOP) so you can physically move the laptop as if it were the robot.

**Runtime:** Runtime → Change runtime type → **T4 or A100 GPU**. Then run cells top to bottom.

**Reference photo:** Uses `references/ref.jpg` and `references/ctx.jpg` from the repo. Commit new ones and re-clone if you want to swap.

## 1. Clone the repo + install deps

In [ ]:
!git clone https://github.com/Ndgandhi23/RU-IEEE-Hackathon.git 2>/dev/null || echo 'repo already cloned'
%cd RU-IEEE-Hackathon

!pip install -q git+https://github.com/huggingface/transformers@main accelerate bitsandbytes ultralytics
!pip install -q --force-reinstall --no-deps Pillow==10.4.0
print('\ndone')

## 2. Load the brain (YOLO + Qwen3-VL)

In [ ]:
import sys, time
from pathlib import Path

import cv2
import numpy as np
import torch

REPO = Path.cwd()
sys.path.insert(0, str(REPO))

assert torch.cuda.is_available(), 'no GPU - Runtime -> Change runtime type -> GPU'
print(f'GPU: {torch.cuda.get_device_name(0)}')

from brain.perception.yolo_finder import YoloFinder
from brain.perception.vlm_scout import VLMScout
from brain.control.loop import Action, ApproachController

print('loading YOLO...')
finder = YoloFinder(weights=str(REPO / 'models' / 'trash_v1.pt'), min_conf=0.5)

print('loading Qwen3-VL-8B (4-bit)...')
t0 = time.monotonic()
scout = VLMScout(load_in_4bit=True)
print(f'  loaded in {time.monotonic() - t0:.1f}s')

## 3. Bind the reference photo (the "target")

In [ ]:
REF = REPO / 'references' / 'ref.jpg'
CTX = REPO / 'references' / 'ctx.jpg'
assert REF.exists() and CTX.exists(), 'missing reference files'

ref_bgr = cv2.imread(str(REF))

# Thumbnail for the corner of each live frame.
def _fit(img, max_side=200):
    h, w = img.shape[:2]
    s = max_side / max(h, w)
    return img if s >= 1.0 else cv2.resize(img, (int(w * s), int(h * s)), interpolation=cv2.INTER_AREA)

ref_thumb = _fit(ref_bgr, 200)

controller = ApproachController(
    target_finder=finder,
    vlm_scout=scout,
    reference_photo=str(REF),
    reporter_photo=str(CTX),
)
print('mission armed - brain is hunting for this target:')

from IPython.display import Image, display
_, buf = cv2.imencode('.jpg', ref_thumb)
display(Image(data=buf.tobytes()))

## 4. Overlay helper (draws bounding boxes, action label, reference thumbnail)

In [ ]:
_COLOR_FORWARD = (0, 220, 0)
_COLOR_TURN    = (0, 220, 220)
_COLOR_STOP    = (0, 0, 220)
_COLOR_SEARCH  = (220, 0, 220)
_COLOR_BBOX    = (255, 255, 0)

_ACTION_COLOR = {
    Action.FORWARD:       _COLOR_FORWARD,
    Action.LEFT:          _COLOR_TURN,
    Action.RIGHT:         _COLOR_TURN,
    Action.STOP:          _COLOR_STOP,
    Action.SEARCH_LEFT:   _COLOR_SEARCH,
    Action.SEARCH_RIGHT:  _COLOR_SEARCH,
    Action.SCOOP_FORWARD: (255, 140, 0),
    Action.BACKUP:        (120, 120, 255),
}

_ACTION_HINT = {
    Action.FORWARD:       'move laptop forward',
    Action.LEFT:          'rotate laptop left',
    Action.RIGHT:         'rotate laptop right',
    Action.STOP:          'hold still',
    Action.SEARCH_LEFT:   'scanning left',
    Action.SEARCH_RIGHT:  'scanning right',
    Action.SCOOP_FORWARD: 'scoop forward',
    Action.BACKUP:        'back up',
}

def overlay(frame, ref_thumb, phase, action, top_detection, fps):
    h, w = frame.shape[:2]
    color = _ACTION_COLOR[action]
    cv2.rectangle(frame, (0, 0), (w - 1, h - 1), color, 14)

    if top_detection is not None:
        x1, y1, x2, y2 = top_detection.xyxy
        cv2.rectangle(frame, (x1, y1), (x2, y2), _COLOR_BBOX, 3)
        label = f'{top_detection.class_name} {top_detection.confidence:.2f}'
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
        cv2.rectangle(frame, (x1, y1 - th - 8), (x1 + tw + 6, y1), _COLOR_BBOX, -1)
        cv2.putText(frame, label, (x1 + 3, y1 - 4), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 2)

    text = action.name
    (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 2.0, 4)
    tx = (w - tw) // 2
    ty = th + 40
    cv2.rectangle(frame, (tx - 18, ty - th - 18), (tx + tw + 18, ty + 18), (0, 0, 0), -1)
    cv2.putText(frame, text, (tx, ty), cv2.FONT_HERSHEY_SIMPLEX, 2.0, color, 4)

    hint = _ACTION_HINT[action]
    (hw, hh), _ = cv2.getTextSize(hint, cv2.FONT_HERSHEY_SIMPLEX, 0.7, 2)
    hx = (w - hw) // 2
    hy = ty + hh + 22
    cv2.putText(frame, hint, (hx, hy), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

    cv2.putText(frame, f'phase: {phase}', (16, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)
    cv2.putText(frame, f'{fps:.1f} fps', (16, h - 18), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)

    th_h, th_w = ref_thumb.shape[:2]
    pad = 16
    x0 = w - th_w - pad
    y0 = h - th_h - pad
    cv2.rectangle(frame, (x0 - 4, y0 - 4), (x0 + th_w + 4, y0 + th_h + 4), (255, 255, 255), 2)
    frame[y0:y0 + th_h, x0:x0 + th_w] = ref_thumb
    cv2.putText(frame, 'target', (x0, y0 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

print('overlay helper ready')

## 5. Browser webcam helper (Colab-only)

Colab runs in your browser, not on the VM. To get your laptop's camera into Python we run a small JavaScript snippet that captures a frame from the browser and hands it back as a base64 image.

In [ ]:
from IPython.display import display, Javascript, Image
from google.colab.output import eval_js
from base64 import b64decode

# NOTE: We do NOT register the JS here. Colab scopes each cell's JS to its own
# output iframe, so JS defined in cell 5 isn't visible from cell 6's eval_js.
# Instead, cell 6 defines AND uses the JS in its own cell (same iframe),
# and uses display_id.update() instead of clear_output to avoid wiping it.

def _decode_frame(data_url):
    jpg = b64decode(data_url.split(',', 1)[1])
    arr = np.frombuffer(jpg, dtype=np.uint8)
    return cv2.imdecode(arr, cv2.IMREAD_COLOR)

print('helpers ready')

## 6. Run the live demo

Allow camera access when prompted. The cell below will stream frames from your browser, run the brain on each, and display the overlaid frame inline. Frames update ~1-3 fps (Colab round-trip is the bottleneck, not the GPU).

Set `NUM_FRAMES` lower for a quick test or higher for a longer demo. Interrupt the cell to stop early.

In [ ]:
from IPython.display import display, Javascript, Image as IPyImage

NUM_FRAMES = 120

# Define the JS in THIS cell so eval_js below runs in the same output iframe.
# All functions attach to window.* so the IIFE wrapper doesn't hide them.
_JS = '''
window.butters_initCam = async function(quality) {
  if (window._butters && window._butters.stream) { return 'already running'; }
  const div = document.createElement('div');
  div.id = 'butters_cam_div';
  div.style.cssText = 'position:fixed;bottom:8px;right:8px;z-index:99999;border:2px solid #0a0;background:#000;';
  const video = document.createElement('video');
  video.style.display = 'block';
  video.width = 320;
  const stream = await navigator.mediaDevices.getUserMedia({video: {width: 640, height: 480}});
  document.body.appendChild(div);
  div.appendChild(video);
  video.srcObject = stream;
  await video.play();
  window._butters = {video, stream, quality};
  return 'started';
};
window.butters_grab = async function() {
  if (!window._butters || !window._butters.video) { throw new Error('camera not initialized'); }
  const v = window._butters.video;
  if (!v.videoWidth) { return null; }
  const canvas = document.createElement('canvas');
  canvas.width = v.videoWidth;
  canvas.height = v.videoHeight;
  canvas.getContext('2d').drawImage(v, 0, 0);
  return canvas.toDataURL('image/jpeg', window._butters.quality);
};
window.butters_stopCam = async function() {
  if (window._butters && window._butters.stream) {
    window._butters.stream.getTracks().forEach(t => t.stop());
    const div = document.getElementById('butters_cam_div');
    if (div) div.remove();
    delete window._butters;
  }
  return 'stopped';
};
'''
display(Javascript(_JS))

print('starting camera (allow access when prompted)...')
cam_status = eval_js('butters_initCam(0.8)')
print(f'cam: {cam_status}')
time.sleep(1.5)  # let video warm up so videoWidth is populated

# Persistent display handles — update in place, never clear_output.
img_handle = display(IPyImage(data=b''), display_id='butters_frame')
log_handle = display({'text/plain': 'starting...'}, raw=True, display_id='butters_log')

t_prev = time.monotonic()
fps_window = []
history = []

try:
    for i in range(NUM_FRAMES):
        try:
            data_url = eval_js('butters_grab()')
        except Exception as e:
            print(f'grab error on frame {i+1}: {e}')
            break
        if not data_url:
            time.sleep(0.1)  # camera not ready yet; retry
            continue
        frame = _decode_frame(data_url)
        if frame is None:
            continue

        detections = finder.detect(frame)
        action = controller.step(frame)

        now = time.monotonic()
        fps_window.append(now - t_prev)
        t_prev = now
        if len(fps_window) > 20:
            fps_window.pop(0)
        fps = len(fps_window) / sum(fps_window) if fps_window else 0.0

        top = detections[0] if detections else None
        overlay(frame, ref_thumb, controller.phase.value, action, top, fps)
        history.append((action.name, len(detections)))

        _, buf = cv2.imencode('.jpg', frame, [cv2.IMWRITE_JPEG_QUALITY, 85])
        img_handle.update(IPyImage(data=buf.tobytes()))
        log_handle.update({
            'text/plain': (
                f'frame {i+1}/{NUM_FRAMES}  phase={controller.phase.value}  '
                f'action={action.name}  dets={len(detections)}  {fps:.1f} fps'
            )
        }, raw=True)
finally:
    try:
        eval_js('butters_stopCam()')
    except Exception:
        pass

print(f'\ndone. actions seen: {sorted(set(a for a, _ in history))}')

## 7. Reset (run a new "mission")

Clears controller state + history so you can rerun the demo with a fresh search-burst counter.

In [ ]:
history.clear()
controller._search_remaining = 0
controller._search_direction = None
print('controller reset. Rerun cell 6 to start again.')